# CellVision-QC — End-to-End Demonstration

This notebook demonstrates the full CellVision-QC workflow using **purely synthetic data**.
No real biological images are required.

**Workflow overview:**
1. Generate synthetic fluorescence microscopy images with simulated healthy / unhealthy phenotypes
2. Preprocess images (background subtraction, normalization)
3. Segment cells using two methods: *Otsu thresholding* and *watershed*
4. Compute segmentation QC metrics
5. Extract per-object morphological and intensity features
6. Train a classifier and assess how segmentation choice affects phenotype discrimination

> **Disclaimer:** All data are synthetic. Results reflect workflow structure, not biological conclusions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({'figure.dpi': 120})

# Install the package in editable mode if running in Colab / fresh environment:
# !pip install -e .. --quiet

## 1. Generate synthetic demo data

In [ ]:
from cellvision_qc.data.synthetic import generate_demo_dataset

DATA_DIR = Path("../data/demo_nb")

generate_demo_dataset(
    output_dir=DATA_DIR,
    n_images=12,
    image_size=256,
    cells_per_image=20,
    healthy_fraction=0.5,
    seed=0,
)

labels_df = pd.read_csv(DATA_DIR / 'labels.csv')
labels_df.head()

## 2. Inspect raw images

In [ ]:
import tifffile

image_paths = sorted((DATA_DIR / 'images').glob('*.tif'))
label_map = dict(zip(labels_df['filename'], labels_df['phenotype']))

# Show one healthy and one unhealthy image side by side
healthy_path = next(p for p in image_paths if label_map[p.name] == 'healthy')
unhealthy_path = next(p for p in image_paths if label_map[p.name] == 'unhealthy')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, path, title in zip(axes, [healthy_path, unhealthy_path], ['Healthy (synthetic)', 'Unhealthy (synthetic)']):
    img = tifffile.imread(str(path)).astype(float) / 65535
    ax.imshow(img, cmap='gray', vmin=0, vmax=0.5)
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.suptitle('Raw synthetic images (16-bit TIFF, scaled for display)', y=1.01)
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
from cellvision_qc.preprocessing import PreprocessingConfig, load_and_preprocess

config = PreprocessingConfig(background_radius=50, gaussian_sigma=None)
preprocessed = load_and_preprocess(healthy_path, config)

raw = tifffile.imread(str(healthy_path)).astype(float) / 65535

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(raw, cmap='gray', vmin=0, vmax=raw.max())
axes[0].set_title('Raw', fontweight='bold')
axes[0].axis('off')
axes[1].imshow(preprocessed, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Preprocessed (background subtracted + normalized)', fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 4. Segmentation comparison

In [ ]:
from cellvision_qc.segmentation.threshold import ThresholdSegmenter
from cellvision_qc.segmentation.watershed import WatershedSegmenter
from skimage.color import label2rgb
from skimage.segmentation import find_boundaries

thresh_seg = ThresholdSegmenter(min_object_size=50)
watershed_seg = WatershedSegmenter(min_distance=10, min_object_size=50)

result_thresh = thresh_seg.segment(preprocessed)
result_ws = watershed_seg.segment(preprocessed)

def overlay(image, labels, alpha=0.4):
    rgb = np.stack([image]*3, axis=-1)
    ov = label2rgb(labels, image=rgb, alpha=alpha, bg_label=0)
    bounds = find_boundaries(labels, mode='outer')
    ov[bounds] = 1.0
    return ov

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(preprocessed, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Preprocessed', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(overlay(preprocessed, result_thresh.label_image))
axes[1].set_title(f'Threshold  ({result_thresh.n_objects} objects)', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(overlay(preprocessed, result_ws.label_image))
axes[2].set_title(f'Watershed  ({result_ws.n_objects} objects)', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Segmentation method comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. QC metrics

In [ ]:
from cellvision_qc.metrics.qc import compute_qc_metrics

qc_thresh = compute_qc_metrics(result_thresh, image_name=healthy_path.name)
qc_ws = compute_qc_metrics(result_ws, image_name=healthy_path.name)

qc_df = pd.DataFrame([qc_thresh.to_dict(), qc_ws.to_dict()])
qc_df[['method', 'n_objects', 'mean_area', 'cv_area', 'coverage_fraction',
       'mean_eccentricity', 'mean_solidity', 'n_border_objects']]

## 6. Feature extraction

In [ ]:
from cellvision_qc.features.extraction import extract_features, aggregate_features

all_features = []
for img_path in image_paths:
    phenotype = label_map[img_path.name]
    prep = load_and_preprocess(img_path, config)
    for seg, method in [(thresh_seg, 'threshold'), (watershed_seg, 'watershed')]:
        result = seg.segment(prep)
        feats = extract_features(result, prep, image_name=img_path.name,
                                 label_column='phenotype', label_value=phenotype)
        all_features.append(feats)

features_df = aggregate_features(all_features)
print(f"Total objects extracted: {len(features_df)}")
features_df.head()

## 7. Feature distributions by phenotype

In [ ]:
import seaborn as sns

feature_cols = ['area', 'eccentricity', 'solidity', 'mean_intensity']
thresh_feats = features_df[features_df['method'] == 'threshold']

fig, axes = plt.subplots(1, len(feature_cols), figsize=(14, 4))
for ax, feat in zip(axes, feature_cols):
    sns.violinplot(data=thresh_feats, x='phenotype', y=feat, ax=ax,
                   palette='Set2', inner='box', linewidth=0.8)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('')
plt.suptitle('Feature distributions by phenotype  |  threshold segmentation',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Phenotype classification

In [ ]:
from cellvision_qc.phenotype.analysis import PhenotypeClassifier

clf = PhenotypeClassifier(classifier_type='random_forest', cv_folds=5)
reports = []

for method in ['threshold', 'watershed']:
    method_df = features_df[features_df['method'] == method]
    report = clf.evaluate(method_df, label_col='phenotype', method_name=method)
    reports.append(report.to_dict())
    print(f"{method:12s}  accuracy={report.accuracy_mean:.3f}±{report.accuracy_std:.3f}  "
          f"F1={report.f1_mean:.3f}  ROC-AUC={report.roc_auc_mean:.3f}")

summary_df = pd.DataFrame(reports)
summary_df[['method', 'accuracy_mean', 'accuracy_std', 'f1_mean', 'roc_auc_mean']]

## 9. Performance comparison bar chart

In [ ]:
from cellvision_qc.visualization.plots import plot_comparison_bar

bar_path = Path('../results/nb_comparison.png')
bar_path.parent.mkdir(parents=True, exist_ok=True)
plot_comparison_bar(summary_df, bar_path)

img_display = plt.imread(str(bar_path))
plt.figure(figsize=(9, 4))
plt.imshow(img_display)
plt.axis('off')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

| Step | Tool |
|------|------|
| Synthetic data generation | `cellvision_qc.data.synthetic` |
| Image preprocessing | `cellvision_qc.preprocessing` |
| Threshold segmentation | `ThresholdSegmenter` |
| Watershed segmentation | `WatershedSegmenter` |
| QC metrics | `cellvision_qc.metrics.qc` |
| Feature extraction | `cellvision_qc.features.extraction` |
| Phenotype classification | `PhenotypeClassifier` |

The comparison shows whether segmentation method choice changes downstream
classification performance — a key practical question in high-content screening
pipeline development.

> All results are based on synthetic data and should not be interpreted as
> biological findings.